In [20]:
import pandas as pd
import numpy as np
import os
from tensorflow import keras
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional, Dropout
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.callbacks import EarlyStopping
import pickle
import joblib
from sklearn.metrics import classification_report

Load data from file

In [11]:
base_dir = os.getcwd()
parent_dir = os.path.dirname(base_dir)
data_train_path = os.path.join(parent_dir, "data", "processed", "training.csv")
data_val_path = os.path.join(parent_dir, "data", "processed", "val.csv")
model_path = os.path.join(parent_dir, "models")

df_train = pd.read_csv(data_train_path)
df_val = pd.read_csv(data_val_path)

X_train = df_train['text']
y_train = df_train['label']
X_val = df_val['text']
y_val = df_val['label']

In [3]:
vocab_size = 10000
embedding_dim = 64
max_length = 50
num_classes = 3

In [4]:
#Tokenize and vectorize
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq   = tokenizer.texts_to_sequences(X_val)

X_train_seq = pad_sequences(X_train_seq, maxlen=max_length, padding='post')
X_val_seq   = pad_sequences(X_val_seq, maxlen=max_length, padding='post')

In [5]:
model = keras.Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_length),
    Bidirectional(LSTM(64, dropout=0.3, recurrent_dropout=0.3)),
    Dropout(0.5), # Adding dropout to avoid overfitting
    Dense(3, activation='softmax')
])

d:\DOCUMENT\School\NLP\external\project\twitter_sentiment_classification\.venv1\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [6]:
model.compile(
    loss='sparse_categorical_crossentropy', # using labels = [0, 1, 2]
    optimizer='adam', 
    metrics=['accuracy'])

In [7]:
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

In [8]:
model.fit(
    X_train_seq, y_train, 
    epochs=10, 
    batch_size=32, 
    validation_data=(X_val_seq, y_val),
    callbacks=[early_stop]
)

Epoch 1/10
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 81s 34ms/step - accuracy: 0.6778 - loss: 0.7282 - val_accuracy: 0.8398 - val_loss: 0.4186
Epoch 2/10
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 75s 33ms/step - accuracy: 0.8028 - loss: 0.4913 - val_accuracy: 0.8909 - val_loss: 0.3112
Epoch 3/10
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 74s 33ms/step - accuracy: 0.8407 - loss: 0.3995 - val_accuracy: 0.9099 - val_loss: 0.2573
Epoch 4/10
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 71s 32ms/step - accuracy: 0.8638 - loss: 0.3473 - val_accuracy: 0.9219 - val_loss: 0.2522
Epoch 5/10
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 66s 29ms/step - accuracy: 0.8771 - loss: 0.3113 - val_accuracy: 0.9319 - val_loss: 0.2206
Epoch 6/10
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 70s 31ms/step - accuracy: 0.8864 - loss: 0.2868 - val_accuracy: 0.9339 - val_loss: 0.2149
Epoch 7/10
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 69s 30ms/step - accuracy: 0.8949 - loss: 0.2655 - val_accuracy: 0.9329 - val_loss: 0.2104
Epoch 8/10
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 108s 48ms/step - accuracy: 0.9031 

In [17]:
lstm_model_path = os.path.join(model_path, "model_bilstm.keras")
model.save(lstm_model_path)

tokenizer_path = os.path.join(model_path, "tokenizer.pkl")
joblib.dump(tokenizer, tokenizer_path)

['d:\\DOCUMENT\\School\\NLP\\external\\project\\twitter_sentiment_classification\\models\\tokenizer.pkl']

In [18]:
le = joblib.load(os.path.join(model_path, "label_encoder.pkl"))

d:\DOCUMENT\School\NLP\external\project\twitter_sentiment_classification\.venv1\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [21]:
y_pred_probs = model.predict(X_val_seq)
y_pred_indices = np.argmax(y_pred_probs, axis=1)

# 3. Chuyển từ index (0, 1, 2) về lại nhãn chữ ban đầu
y_pred_labels = le.inverse_transform(y_pred_indices)

# 4. In báo cáo (Lưu ý: y_val cũng phải cùng định dạng với y_pred_labels)
# Nếu y_val đang là số, hãy so sánh nó với y_pred_indices
print(classification_report(y_val, y_pred_indices))

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
              precision    recall  f1-score   support

           0       0.95      0.94      0.95       265
           1       0.96      0.96      0.96       457
           2       0.92      0.93      0.93       277

    accuracy                           0.95       999
   macro avg       0.94      0.94      0.94       999
weighted avg       0.95      0.95      0.95       999

